# Notebook 30 — Eval & Measurement Framework

The `eval` module closes the most critical gap in agentic frameworks: **how do you know your agent is actually doing a good job?**

| Class | Purpose |
|---|---|
| `EvalCase` | single (input, expected) test case |
| `EvalDataset` | collection of cases with filtering/sampling |
| `Evaluator` | runs an agent against a dataset, concurrently |
| `EvalReport` | aggregated stats (mean, p50, p95 per metric) |
| `EvalSuite` | batch-run multiple evaluators |
| `Benchmark` | head-to-head comparison of two agents |
| `EvalRegistry` | register named datasets and metrics |
| Built-in metrics | `ExactMatch`, `ContainsMatch`, `RegexMatch`, `F1Score`, `JSONFieldMatch`, `Latency`, `TokenCount`, `CostEstimate`, `LLMJudge` |

## 1. Setup

In [ ]:
import asyncio
from multigen.eval import (
    EvalCase, EvalDataset, Evaluator, EvalSuite, EvalRegistry,
    ExactMatch, ContainsMatch, RegexMatch, JSONFieldMatch,
    F1Score, Latency, TokenCount, CostEstimate, LLMJudge, CustomMetric,
    Benchmark, get_eval_registry,
)
print('eval module loaded')

## 2. Building a Dataset

`EvalDataset` is a collection of `EvalCase` objects.  Cases can be filtered, sampled, and saved to JSONL.

In [ ]:
# Inline dataset
qa_dataset = EvalDataset([
    EvalCase({'question': 'What is 2+2?'},         expected='4',             tags=['math', 'easy']),
    EvalCase({'question': 'Capital of France?'},    expected='Paris',         tags=['geo', 'easy']),
    EvalCase({'question': 'Who wrote Hamlet?'},     expected='Shakespeare',   tags=['lit', 'medium']),
    EvalCase({'question': 'Square root of 144?'},  expected='12',            tags=['math', 'medium']),
    EvalCase({'question': 'Speed of light (km/s)?'}, expected='299792',      tags=['physics', 'hard']),
])

print(f'Dataset: {qa_dataset}')
print(f'Math subset: {qa_dataset.filter_by_tag("math")}')
print(f'Sample of 3: {qa_dataset.sample(3, seed=42)}')

# Save / load JSONL
qa_dataset.save_jsonl('/tmp/qa.jsonl')
loaded = EvalDataset.load_jsonl('/tmp/qa.jsonl')
print(f'Loaded from JSONL: {loaded}')

## 3. Running Your First Evaluation

An `Evaluator` runs *any async or sync callable* against the dataset.

In [ ]:
# A simple mock agent
async def answer_agent(inp: dict) -> str:
    import asyncio
    answers = {
        'What is 2+2?': '4',
        'Capital of France?': 'The capital of France is Paris.',
        'Who wrote Hamlet?': 'William Shakespeare',
        'Square root of 144?': '12',
        'Speed of light (km/s)?': '299792',
    }
    await asyncio.sleep(0.01)   # simulate LLM latency
    return answers.get(inp['question'], 'I do not know.')

# Evaluate with multiple metrics
metrics = [ExactMatch(), ContainsMatch(), F1Score(), Latency(), TokenCount()]

ev = Evaluator(
    agent=answer_agent,
    dataset=qa_dataset,
    metrics=metrics,
    agent_name='AnswerBot v1',
    concurrency=5,
)

report = await ev.run()
print(report.summary())

## 4. Digging Into Results

In [ ]:
stats = report.stats()
print('error_rate:', stats['error_rate'])
print('exact_match mean:', stats['metrics']['exact_match']['mean'])
print('contains_match mean:', stats['metrics']['contains_match']['mean'])
print('f1_score p50:', stats['metrics']['f1_score']['p50'])
print()
# Per-case results
for r in report.results:
    em = r.scores.get('exact_match', 0)
    cm = r.scores.get('contains_match', 0)
    print(f"[{r.case_id}] EM={em:.0f} CM={cm:.0f}  '{r.output}'  (expected: '{r.expected}')")

## 5. Built-in Metrics

### ExactMatch — strict string equality

In [ ]:
em_ci  = ExactMatch(case_sensitive=False)
em_cs  = ExactMatch(case_sensitive=True)

case = EvalCase({'q': 'x'}, expected='Paris')
print('case-insensitive PARIS:', em_ci.score(case, 'PARIS'))   # 1.0
print('case-sensitive   PARIS:', em_cs.score(case, 'PARIS'))   # 0.0
print('case-insensitive paris:', em_ci.score(case, 'paris'))   # 1.0

### ContainsMatch, RegexMatch, JSONFieldMatch

In [ ]:
cm  = ContainsMatch()
rm  = RegexMatch()  # pattern from case.expected
jfm = JSONFieldMatch(fields=['answer'])

# ContainsMatch
c1 = EvalCase({'q': 'x'}, expected='Paris')
print('ContainsMatch (capital is Paris):', cm.score(c1, 'The capital is Paris, France.'))  # 1.0

# RegexMatch — expected IS the pattern
c2 = EvalCase({'q': 'x'}, expected=r'\d{4,6}')
print('RegexMatch (299792):', rm.score(c2, '299792'))   # 1.0
print('RegexMatch (abc):   ', rm.score(c2, 'abc'))      # 0.0

# JSONFieldMatch
c3 = EvalCase({'q': 'x'}, expected={'answer': 'Paris', 'confidence': 0.95})
out = {'answer': 'Paris', 'confidence': 0.95, 'extra': 'ignored'}
print('JSONFieldMatch:', jfm.score(c3, out))   # 1.0

### F1Score — token-level overlap (common in QA)

In [ ]:
f1 = F1Score()
c = EvalCase({'q': 'x'}, expected='The quick brown fox')
print('F1 (exact):    ', f1.score(c, 'The quick brown fox'))   # 1.0
print('F1 (partial):  ', f1.score(c, 'The quick dog'))          # ~0.5
print('F1 (no match): ', f1.score(c, 'completely wrong'))       # 0.0

### Latency, TokenCount, CostEstimate

In [ ]:
lat  = Latency(budget_ms=1000)
tok  = TokenCount()
cost = CostEstimate(cost_per_1k_tokens=0.002)

c = EvalCase({'q': 'x'}, expected='Paris')
output = 'The capital of France is Paris.'
print('Latency raw ms:    ', lat.score(c, output, latency_ms=120.5))
print('Token count:       ', tok.score(c, output))   # word count proxy
print('Cost estimate USD: ', cost.score(c, output))

## 6. LLM-as-Judge

Plug in any LLM to score agent outputs on subjective criteria (coherence, helpfulness, safety).

In [ ]:
# Mock judge — replace with real LLM call
async def coherence_judge(case, output):
    # Return score in [0, 1]
    length_penalty = min(1.0, len(str(output).split()) / 5)
    return length_penalty

coherence = LLMJudge(judge_fn=coherence_judge, name='coherence')

ev_judged = Evaluator(
    answer_agent, qa_dataset,
    metrics=[ExactMatch(), coherence],
    agent_name='AnswerBot v1 (judged)',
    concurrency=3,
)
report_judged = await ev_judged.run()
print(report_judged.summary())

## 7. Streaming Evaluation

Use `run_streaming()` to process results as they arrive — useful for live dashboards.

In [ ]:
print('Streaming results:')
async for result in ev.run_streaming():
    em = result.scores.get('exact_match', 0)
    lat_ms = result.latency_ms
    print(f'  [{result.case_id}] EM={em:.0f}  lat={lat_ms:.0f}ms  "{result.output}"')

## 8. EvalSuite — batch comparison

In [ ]:
async def slow_agent(inp: dict) -> str:
    import asyncio
    await asyncio.sleep(0.05)
    answers = {
        'What is 2+2?': '4',
        'Capital of France?': 'Paris',
        'Who wrote Hamlet?': 'Shakespeare',
        'Square root of 144?': '12',
        'Speed of light (km/s)?': 'about 300000',  # wrong!
    }
    return answers.get(inp['question'], 'unknown')

suite = EvalSuite('AnswerBot comparison')
suite.add('v1-fast', Evaluator(answer_agent, qa_dataset, [ExactMatch(), Latency()]))
suite.add('v2-slow', Evaluator(slow_agent,   qa_dataset, [ExactMatch(), Latency()]))

reports = await suite.run()
suite.print_comparison(reports)

## 9. Head-to-Head Benchmark

In [ ]:
bench = Benchmark(
    agent_a=answer_agent,
    agent_b=slow_agent,
    dataset=qa_dataset,
    metrics=[ExactMatch(), ContainsMatch(), F1Score(), Latency(), TokenCount()],
    label_a='fast-v1',
    label_b='slow-v2',
    concurrency=4,
)

result = await bench.run()
print(result.summary())
print()
for m in ['exact_match', 'f1_score', 'latency']:
    w = result.winner(m)
    d = result.delta(m)
    print(f'{m:<20} winner={w:<10} delta={d:+.4f}')

## 10. EvalRegistry — reusable datasets and metrics

In [ ]:
registry = get_eval_registry()

# Register once
registry.register_dataset('qa_basic', qa_dataset)
registry.register_metric('em',  ExactMatch())
registry.register_metric('cm',  ContainsMatch())
registry.register_metric('f1',  F1Score())
registry.register_metric('lat', Latency())

print('Registered datasets:', registry.list_datasets())
print('Registered metrics: ', registry.list_metrics())

# Retrieve by name in any other module
ds  = registry.get_dataset('qa_basic')
em  = registry.get_metric('em')
print(f'Retrieved dataset: {ds}, metric: {em}')

## 11. Saving and Loading Reports

In [ ]:
# Save full report to JSONL
report.save_jsonl('/tmp/eval_report.jsonl')

# Reload later
from multigen.eval import EvalReport
loaded_report = EvalReport.load_jsonl('/tmp/eval_report.jsonl')
print('Loaded report:', loaded_report)
print(loaded_report.summary())

## 12. Custom Metric

In [ ]:
def word_count_ratio(case, output, latency_ms):
    """Score = min(1, output_words / expected_words)  — penalises brevity."""
    exp_words = len(str(case.expected).split())
    out_words = len(str(output).split())
    return min(1.0, out_words / max(1, exp_words))

verbosity = CustomMetric(fn=word_count_ratio, name='verbosity_ratio')

ev_v = Evaluator(answer_agent, qa_dataset, metrics=[verbosity], agent_name='verbosity-test')
rep_v = await ev_v.run()
print(rep_v.summary())

## 13. Error Handling

The `Evaluator` catches exceptions and timeouts per-case; it never aborts the full run.

In [ ]:
async def flaky_agent(inp):
    import asyncio
    if 'Hamlet' in inp.get('question', ''):
        raise ValueError('Knowledge base offline!')
    return 'some answer'

ev_err = Evaluator(
    flaky_agent, qa_dataset,
    metrics=[ExactMatch(), Latency()],
    agent_name='flaky',
    concurrency=2,
)
rep_err = await ev_err.run()
print(rep_err.summary())
print()
errors = [r for r in rep_err.results if r.error]
for e in errors:
    print(f'  Case {e.case_id}: {e.error}')

## Summary

| Component | Description |
|---|---|
| `EvalCase` | `(input, expected, tags, metadata)` |
| `EvalDataset` | collection + filter/sample/JSONL I/O |
| `Evaluator` | concurrent agent runner + per-case scorer |
| `EvalReport` | mean/p50/p95 per metric + JSONL save/load |
| `EvalSuite` | batch-run N evaluators, print comparison |
| `Benchmark` | A vs B head-to-head on same dataset |
| `EvalRegistry` | global registry of datasets + metrics |
| `ExactMatch` | strict string equality |
| `ContainsMatch` | substring containment |
| `RegexMatch` | regex pattern search |
| `JSONFieldMatch` | key-level dict comparison |
| `F1Score` | token-level F1 (QA-style) |
| `Latency` | raw ms (p50/p95 in report) |
| `TokenCount` | whitespace token count |
| `CostEstimate` | USD proxy from token counts |
| `LLMJudge` | pluggable async LLM-as-judge |
| `CustomMetric` | wrap any callable as a metric |